# AeroIntel — 01 Data Prep (Colab)

Implements spec **A6 steps 2–4**: audit → remap → dedupe + leakage-safe split.

**Run top to bottom.** Everything lands in Google Drive under `AeroIntel/` so nothing is lost when the runtime disconnects:

```
/content/drive/MyDrive/AeroIntel/
  datasets/raw/<source>/           # raw downloads (untouched)
  datasets/merged/                 # remapped 4-class YOLO dataset (output)
  datasets/audit/<source>/         # audit JSON + contact sheets
  datasets/field_test/             # >=50 hand-collected images (spec A6)
```

Before running, set the dataset sources in the **Config** cell. Fill `ml/class_map.yaml` mappings in the Remap cell after reading the audit.

In [ ]:
# @title 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/AeroIntel'
import os
for d in ['datasets/raw', 'datasets/raw_uploads', 'datasets/merged',
           'datasets/audit', 'datasets/field_test', 'runs', 'models', 'logs']:
    os.makedirs(os.path.join(DRIVE_ROOT, d), exist_ok=True)
print('Drive ready at', DRIVE_ROOT)

In [ ]:
# @title 2. Config — set your dataset sources here
ROBOFLOW_API_KEY = ''  # @param {type:"string"}  # get from app.roboflow.com -> Settings

# One entry per Roboflow dataset. name = folder name, must match the keys in ml/class_map.yaml
ROBOFLOW_DATASETS = [
    # {'name': 'roboflow_aircraft_corrosion', 'workspace': 'YOUR_WORKSPACE',
    #  'project': 'YOUR_PROJECT', 'version': 2, 'fmt': 'yolov8'},
]

# Non-Roboflow datasets: upload the ORIGINAL zip to
# Drive: AeroIntel/datasets/raw_uploads/<name>.zip   (name must match class_map keys)
UPLOADED_ZIPS = []  # e.g. ['dataset_b', 'dataset_c']

print('Configured Roboflow sources:', [d['name'] for d in ROBOFLOW_DATASETS])
print('Configured uploaded zips  :', UPLOADED_ZIPS)

In [ ]:
# @title 3. Download / unpack sources into Drive (raw, untouched)
import os, zipfile, datetime
from pathlib import Path

RAW = Path(DRIVE_ROOT) / 'datasets/raw'
UPLOADS = Path(DRIVE_ROOT) / 'datasets/raw_uploads'
log_path = RAW / 'download_log.txt'

def log(msg):
    line = f"{datetime.datetime.now().isoformat()} | {msg}"
    print(line)
    with open(log_path, 'a') as f:
        f.write(line + '\n')

# Roboflow downloads
if ROBOFLOW_DATASETS:
    from roboflow import Roboflow
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
for ds in ROBOFLOW_DATASETS:
    out = RAW / ds['name']
    if out.exists() and any(out.iterdir()):
        log(f"SKIP {ds['name']} (already in Drive)")
        continue
    project = rf.workspace(ds['workspace']).project(ds['project'])
    version = project.version(ds['version'])
    tmp = version.download('yolov8' if ds.get('fmt', 'yolov8') == 'yolov8' else ds['fmt'])
    os.makedirs(out, exist_ok=True)
    os.system(f"mv {tmp}/* '{out}/'")
    log(f"DOWNLOADED {ds['name']} v{ds['version']} -> {out}")

# Uploaded zips
for name in UPLOADED_ZIPS:
    out = RAW / name
    if out.exists() and any(out.iterdir()):
        log(f"SKIP {name} (already in Drive)")
        continue
    z = UPLOADS / f'{name}.zip'
    assert z.exists(), f'Missing {z} — upload it to Drive first'
    out.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(z) as f:
        f.extractall(out)
    log(f"UNPACKED {name} -> {out}")

print('\nRaw sources in Drive:')
for p in sorted(RAW.iterdir()):
    if p.is_dir(): print(' -', p.name)

In [ ]:
# @title 4. Audit every source (spec A6 step 2) — LOOK at the contact sheets
import json, random, glob
from pathlib import Path
from collections import Counter
import cv2
import numpy as np
from PIL import Image

RAW = Path(DRIVE_ROOT) / 'datasets/raw'
AUDIT = Path(DRIVE_ROOT) / 'datasets/audit'
AUDIT.mkdir(parents=True, exist_ok=True)
random.seed(42)

def find_split_root(src: Path):
    """Locate the folder that contains train/valid|val/test."""
    if all((src / s).exists() for s in ('train', 'valid')):
        return src
    for child in src.iterdir():
        if child.is_dir() and all((child / s).exists() for s in ('train', 'valid', 'test')):
            return child
    return src

def audit_source(src: Path):
    root = find_split_root(src)
    names = {}
    for y in root.rglob('*.yaml'):
        try:
            for i, n in enumerate(str(y.read_text()).split('names:')[1].split()):
                if i % 2 == 0 and n.isdigit():
                    names[int(n)] = str(y.read_text()).split('names:')[1].split()[i + 1]
        except Exception:
            pass
    stats = {'source': src.name, 'splits': {}, 'class_counts': Counter(),
             'bbox_sizes': [], 'missing_labels': 0, 'bad_labels': 0}
    for split in ['train', 'valid', 'val', 'test']:
        img_dir = root / split / 'images'
        if not img_dir.exists():
            img_dir = root / split
        if not img_dir.exists():
            continue
        images = [p for p in img_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}]
        lbl_dir = root / split / 'labels' if (root / split / 'labels').exists() else img_dir
        n_boxes = 0
        for img in images:
            lbl = (lbl_dir / (img.stem + '.txt'))
            if not lbl.exists():
                stats['missing_labels'] += 1
                continue
            rows = [r.split() for r in lbl.read_text().splitlines() if r.strip()]
            for r in rows:
                if len(r) < 5:
                    stats['bad_labels'] += 1
                    continue
                c = int(float(r[0])); w, h = float(r[3]), float(r[4])
                stats['class_counts'][c] += 1
                stats['bbox_sizes'].append(round(w * h, 6))
                n_boxes += 1
        stats['splits'][split] = {'images': len(images), 'boxes': n_boxes}

    # contact sheet of 20 random images with boxes drawn
    all_imgs = sorted(root.rglob('images/*')) if (root / 'train' / 'images').exists() else sorted(root.rglob('*.*'))
    all_imgs = [p for p in all_imgs if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}]
    sample = random.sample(all_imgs, min(20, len(all_imgs)))
    tiles = []
    for p in sample:
        img = cv2.imread(str(p))
        if img is None:
            continue
        h, w = img.shape[:2]
        lbl = p.parent.parent / 'labels' / (p.stem + '.txt') if 'images' in p.parts else p.with_suffix('.txt')
        if lbl.exists():
            for r in [l.split() for l in lbl.read_text().splitlines() if l.strip()]:
                if len(r) < 5: continue
                cx, cy, bw, bh = map(float, r[1:5])
                x1, y1 = int((cx - bw / 2) * w), int((cy - bh / 2) * h)
                x2, y2 = int((cx + bw / 2) * w), int((cy + bh / 2) * h)
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(img, names.get(int(r[0]), str(r[0])), (x1, max(15, y1 - 5)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
        img = cv2.resize(img, (320, 320))
        tiles.append(img)
    if tiles:
        rows_n = 4
        while len(tiles) % rows_n: tiles.append(np.zeros_like(tiles[0]))
        grid = np.vstack([np.hstack(tiles[i:i + rows_n]) for i in range(0, len(tiles), rows_n)])
        out = AUDIT / f"contact_sheet_{src.name}.jpg"
        cv2.imwrite(str(out), grid)
        stats['contact_sheet'] = str(out)

    stats['class_counts'] = {names.get(k, str(k)): v for k, v in sorted(stats['class_counts'].items())}
    stats['bbox_area_percentiles'] = {p: float(np.percentile(stats['bbox_sizes'], p)) if stats['bbox_sizes'] else None
                                      for p in [5, 50, 95]}
    stats['bbox_sizes'] = len(stats['bbox_sizes'])
    stats['names_found'] = names
    return stats

all_stats = {}
for src in sorted(p for p in RAW.iterdir() if p.is_dir() and p.name != 'raw_uploads'):
    s = audit_source(src)
    all_stats[src.name] = s
    print(f"\n=== {src.name} ===")
    print(json.dumps(s, indent=2, default=str))

(AUDIT / 'audit_summary.json').write_text(json.dumps(all_stats, indent=2, default=str))
print('\nAudit saved. OPEN THE CONTACT SHEETS IN DRIVE AND LOOK AT THEM (spec A6: reject datasets with bad labels).')

## Remap
After reviewing the audit, fill the mapping below for each source: `{original_class_id: aerointel_class_name}`.
Unmapped classes are **dropped and logged** (spec A6 step 3). Keep this consistent with `ml/class_map.yaml`.

In [ ]:
# @title 5. Remap to the 4-class schema (spec A6 step 3)
CLASS_NAMES = {0: 'crack', 1: 'corrosion', 2: 'dent', 3: 'missing_fastener'}
NAME_TO_ID = {v: k for k, v in CLASS_NAMES.items()}

# source name -> {original class id: aerointel class name}
REMAP = {
    # 'roboflow_aircraft_corrosion': {0: 'corrosion'},
    # 'dataset_b': {0: 'crack', 3: 'missing_fastener'},
    # 'dataset_c': {0: 'dent'},
}

MERGED = Path(DRIVE_ROOT) / 'datasets/merged'
RAW = Path(DRIVE_ROOT) / 'datasets/raw'
for split in ['train', 'valid', 'test']:
    (MERGED / split / 'images').mkdir(parents=True, exist_ok=True)
    (MERGED / split / 'labels').mkdir(parents=True, exist_ok=True)

drop_log = {}

def find_split_root(src: Path):
    if all((src / s).exists() for s in ('train', 'valid')):
        return src
    for child in src.iterdir():
        if child.is_dir() and all((child / s).exists() for s in ('train', 'valid', 'test')):
            return child
    return src

def split_alias(s):
    return {'val': 'valid'}.get(s, s)

for src in sorted(RAW.iterdir()):
    if not src.is_dir() or src.name not in REMAP:
        continue
    mapping = REMAP[src.name]
    root = find_split_root(src)
    for split_dir in root.iterdir():
        if not split_dir.is_dir() or split_dir.name not in ('train', 'valid', 'val', 'test'):
            continue
        out_split = split_alias(split_dir.name)
        img_dir = split_dir / 'images' if (split_dir / 'images').exists() else split_dir
        lbl_dir = split_dir / 'labels' if (split_dir / 'labels').exists() else img_dir
        n_kept = n_dropped_cls = n_empty = 0
        for img in sorted(img_dir.iterdir()):
            if img.suffix.lower() not in {'.jpg', '.jpeg', '.png'}:
                continue
            lbl = lbl_dir / (img.stem + '.txt')
            if not lbl.exists():
                continue
            out_lines, dropped = [], 0
            for line in lbl.read_text().splitlines():
                parts = line.split()
                if len(parts) < 5:
                    continue
                c = int(float(parts[0]))
                if c not in mapping:
                    dropped += 1
                    continue
                parts[0] = str(NAME_TO_ID[mapping[c]])
                out_lines.append(' '.join(parts))
            key = f"{src.name}:{c if False else 'orig_id'}"
            if dropped:
                drop_log.setdefault(src.name, Counter())[f'class_{c}_dropped_boxes'] = drop_log[src.name].get(f'class_{c}_dropped_boxes', 0) + dropped
            if not out_lines:
                n_empty += 1
                continue  # drop images with no mappable boxes (keeps classes honest)
            stem = f"{src.name}__{img.stem}"
            (MERGED / out_split / 'images' / f"{stem}{img.suffix.lower()}").write_bytes(img.read_bytes())
            (MERGED / out_split / 'labels' / f"{stem}.txt").write_text('\n'.join(out_lines) + '\n')
            n_kept += 1
        print(f"{src.name} [{split_dir.name}] kept={n_kept} empty_skipped={n_empty}")

print('\nDropped boxes per source:', {k: dict(v) for k, v in drop_log.items()} or 'none')

# write data.yaml
data_yaml = MERGED / 'data.yaml'
data_yaml.write_text(
    f"path: {MERGED}\ntrain: train/images\nval: valid/images\ntest: test/images\n"
    f"names:\n" + ''.join(f"  {i}: {n}\n" for i, n in CLASS_NAMES.items())
)
print('Wrote', data_yaml)

In [ ]:
# @title 6. Dedupe (perceptual hash) + leakage-safe check (spec A6 step 4)
!pip -q install ImageHash
import imagehash
from pathlib import Path
from collections import defaultdict

MERGED = Path(DRIVE_ROOT) / 'datasets/merged'
HASH_DIST = 8   # hamming distance <= 8 on 16px dhash = near-duplicate

def phash(p):
    try:
        return imagehash.phash(Image.open(p).convert('RGB'))
    except Exception:
        return None

# group by base stem so augmented variants of one source image share a split later
def base_stem(stem):
    for sep in ('__', '_rf_', '-jpg', '_aug'):
        if sep in stem:
            return stem.split(sep)[0]
    return stem

splits = ['train', 'valid', 'test']
hashes = defaultdict(list)
removed = 0
for split in splits:
    for img in sorted((MERGED / split / 'images').iterdir()):
        h = phash(img)
        if h is None:
            continue
        dup = any((h - hh) <= HASH_DIST for hh, _ in hashes[split])
        if dup and split != 'test':
            img.unlink()
            lbl = MERGED / split / 'labels' / (img.stem + '.txt')
            if lbl.exists(): lbl.unlink()
            removed += 1
        else:
            hashes[split].append((h, base_stem(img.stem)))

print(f'Removed {removed} near-duplicates (within split).')

# leakage check: same base image appearing in both train and test
train_bases = {b for _, b in hashes['train']}
leaks = [b for _, b in hashes['test'] if b in train_bases]
print(f'Leakage check: {len(leaks)} base images appear in BOTH train and test.')
if leaks:
    print('  -> move their test copies out or re-split before training (metrics would be inflated).')
    print('  examples:', leaks[:10])

In [ ]:
# @title 7. Final dataset summary + zip to Drive
import shutil
from pathlib import Path
from collections import Counter

MERGED = Path(DRIVE_ROOT) / 'datasets/merged'
CLASS_NAMES = {0: 'crack', 1: 'corrosion', 2: 'dent', 3: 'missing_fastener'}
summary = {}
for split in ['train', 'valid', 'test']:
    c = Counter()
    for lbl in (MERGED / split / 'labels').glob('*.txt'):
        for line in lbl.read_text().splitlines():
            if line.strip():
                c[CLASS_NAMES.get(int(float(line.split()[0])), '?')] += 1
    n_img = len(list((MERGED / split / 'images').iterdir()))
    summary[split] = {'images': n_img, 'boxes': dict(c)}
    print(f"{split}: {n_img} images, boxes={dict(c)}")

(MERGED / 'dataset_summary.json').write_text(json.dumps(summary, indent=2))

zip_path = '/content/aerointel_dataset_v1'
shutil.make_archive(zip_path, 'zip', MERGED)
!cp {zip_path}.zip "{DRIVE_ROOT}/datasets/"
print('\nZipped to Drive: AeroIntel/datasets/aerointel_dataset_v1.zip')
print('Next: open 02_train_yolo.ipynb')